In [ ]:
# Confirm T4 + RAM before anything. Kaggle free tier sometimes allocates
# a P100 or drops to CPU; this cell catches that in 2 seconds.
!nvidia-smi
!free -h

# reconcile_gst2b_env — real GRPO training (Kaggle T4x2, Qwen3-4B)

60 GRPO steps, Qwen3-4B + LoRA rank 16, via TRL's `environment_factory`
pattern. Eval every 20 steps (0/20/40/60 = 4 eval points). `num_generations=2`
to halve the rollout KV-cache footprint on the 16 GB T4s.

Third training attempt in this branch:
- **Qwen3-0.6B**: bimodal 0.25 tool-call rate; flat 0.353 curve (data/smoke_test_10step.json)
- **Qwen3-1.7B**: zero tool-call rate, degenerate collapse (data/training_log_qwen3_1_7b_partial.json)
- **Qwen3-4B** (this run): Qwen3-4B-Instruct ships with explicit function-calling
  post-training; hypothesis is that 4B's pre-training tool-call prior is strong
  enough to clear the coin-flip^5 chaining wall that 0.6B/1.7B couldn't.

Pipeline-parallel across 2× T4 via `device_map="auto"`. Base 8 GB splits
to ~4 GB/GPU; LoRA + optimizer + 2-generation KV cache adds ~2-3 GB/GPU
at peak. 9-10 GB headroom per GPU. **Expected runtime**: ~4.5 h on T4x2,
well within Kaggle's 9 h session quota. Use **Save & Run All (Commit)**
for background execution.

## 1 — Install deps

TRL from main (environment_factory is experimental and needs >= 0.21).
transformers from main for chat-template + tool-calling fixes. No Unsloth
(conflicts with TRL's native generation) and no explicit xformers pin
(the Kaggle image ships a working one).

In [ ]:
!pip install -q 'trl @ git+https://github.com/huggingface/trl.git@main'
!pip install -q 'transformers @ git+https://github.com/huggingface/transformers.git@main'
!pip install -q peft accelerate bitsandbytes datasets
!pip install -q networkx pandas numpy matplotlib 'pydantic>=2.5'

## 2 — Clone fork and install env package

In [ ]:
import os, sys

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/akashkathole7/OpenEnv.git')
BRANCH = os.environ.get('REPO_BRANCH', 'scaffold/reconcile-gst2b')
CLONE_DIR = '/kaggle/working/OpenEnv'

# Kaggle writable root is /kaggle/working/. Step out before wiping.
%cd /kaggle/working
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {CLONE_DIR}
%cd {CLONE_DIR}

# Install openenv-core + all deps (fastmcp, fastapi, uvicorn, pydantic, httpx, ...).
!pip install -q -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR)
sys.path.insert(0, f'{CLONE_DIR}/src')

# Confirm fix is live.
!cd {CLONE_DIR} && git log --oneline -3
print()
!ls {CLONE_DIR}/envs/reconcile_gst2b_env/scripts/train_grpo_real.py && echo 'train_grpo_real.py present'

## 3 — Run GRPO training

Target ~4.5 h on T4x2 pipeline-parallel. Eval every 20 steps writes
R1/R2/R3/R4/total to `data/real_training/curves.json` after each eval
point (0/20/40/60 = 4 eval points). Checkpoints at step 30 and step 60.

Watch the first 2 training-step log dicts for:
- `completions/mean_terminated_length` — MUST be > 0 (1.7B collapsed to 0 here)
- `tools/call_frequency` — MUST be > 0 (1.7B stayed at 0 every step)
- `entropy` — MUST be > 0.25 (1.7B collapsed to 0.12)

If any of these three show the 1.7B failure signature at step 1-2, halt
via Kaggle's Stop button — the 4B plateau is also confirmed and no point
burning the remaining 4 h quota.

Safety nets baked into the script: absolute 0.30 early-stop on eval
total, grad_norm=0 sanity at step 1, adapter_check at each eval.

In [ ]:
!PYTHONPATH={CLONE_DIR}/src:{CLONE_DIR} python -m envs.reconcile_gst2b_env.scripts.train_grpo_real \
    --output-dir={CLONE_DIR}/data/real_training \
    --total-steps=60 \
    --eval-every=20 \
    --checkpoint-every=30

## 4 — Plot training curves

Expect R3/R4 to climb from near-floor toward 0.9 (Rule 36(4) compliance
and step efficiency are learnable with surface prompting). R1/R2 may
stay near 0.01 — honest finding if so. Flat R3/R4 after 150 steps means
training is broken (no gradient reaching LoRA, generation degenerate,
etc.) — inspect per-step logs in Cell 3.

In [ ]:
import json
import matplotlib.pyplot as plt

curves_path = f'{CLONE_DIR}/data/real_training/curves.json'
with open(curves_path) as f:
    curves = json.load(f)

steps = curves['steps']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for k, style, color in [('R1', '-o', '#d73a49'), ('R2', '-s', '#e65100'),
                         ('R3', '-^', '#2188ff'), ('R4', '-d', '#2e7d32')]:
    axes[0].plot(steps, curves[k], style, label=k, color=color)
axes[0].set_title('per-component reward (eval seeds 9030–9039)')
axes[0].set_xlabel('training step')
axes[0].set_ylabel('component mean')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(-0.05, 1.05)

axes[1].plot(steps, curves['total'], '-o', color='black')
axes[1].set_title('total reward (composite)')
axes[1].set_xlabel('training step')
axes[1].axhline(y=0.179, color='#6699cc', linestyle='--', label='prompted baseline (0.18)')
axes[1].axhline(y=-1.0, color='#d73a49', linestyle='--', label='raw baseline (-1.00)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CLONE_DIR}/data/real_training/curves.png', dpi=120)
plt.show()

print(f"n eval points: {len(steps)}")
print(f"final total: {curves['total'][-1] if curves['total'] else 'none'}")
print(f"config: {curves['config']}")

## 5 — Download curves + final checkpoint

Kaggle keeps `/kaggle/working/` around; use the **Output** tab on the right
side of the notebook to download `data/real_training/curves.json` and the
tar of the final checkpoint.

In [ ]:
import subprocess

# Tar the last checkpoint so it's one clickable output.
last_ckpt_dir = f'{CLONE_DIR}/data/real_training/checkpoints/step_60'
out_tar = '/kaggle/working/final_checkpoint.tar.gz'
subprocess.run(['tar', 'czf', out_tar, '-C', f'{CLONE_DIR}/data/real_training/checkpoints', 'step_60'], check=False)
!ls -la {out_tar}

# Copy curves + plot to /kaggle/working/ so they show up in Output tab.
!cp {CLONE_DIR}/data/real_training/curves.json /kaggle/working/curves.json
!cp {CLONE_DIR}/data/real_training/curves.png /kaggle/working/curves.png 2>/dev/null || true
!ls -la /kaggle/working/*.json /kaggle/working/*.png /kaggle/working/*.tar.gz 2>/dev/null